# Biblical Muse Glimmer 30B DPO Training with Unsloth (4-bit QLoRA)

Phase 2 DPO training on top of the SFT LoRA produced by `biblical_muse_glimmer_30b_sft_unsloth_4bit.ipynb`.

- Base model: `meta-models/Muse-Glimmer-30B` — Meta's own repo, loaded with `use_exact_model_name=True` so Unsloth's mapper cannot substitute its `-unsloth-bnb-4bit` re-pack. Same base, same loading path as the SFT notebook.
- SFT LoRA input: `output/biblical_muse_glimmer_30b_unsloth_4bit/lora_adapters`
- DPO data: `data/training-data/biblical_persona_v2/biblical_personas_v2_dpo.jsonl`
- DPO LoRA output: `output/biblical_muse_glimmer_30b_unsloth_4bit_dpo_v2/lora_adapters` (run 1 wrote to `..._dpo/`; that directory is kept and only read from)

Run this only after the Glimmer SFT notebook has finished and written `lora_adapters/complete.json`. The Setup cell refuses to continue otherwise.

**Written against TRL 1.15.0** (the version the SFT notebook's environment cell installed on 2026-11-09), not the TRL 0.24 the Qwen3.8 DPO notebook used. The DPO trainer was rewritten between them, and four things here differ from the Qwen notebook because of it:

1. `max_prompt_length` no longer exists. Length is filtered in the formatting cell and capped by `max_length` only.
2. The dataset is passed in TRL's **conversational** format (message lists plus a per-row `chat_template_kwargs`), so TRL renders prompts and completions through Glimmer's own chat template. That is what puts the `to=user` channel header and the closing `<|eot|>` inside each completion, matching what SFT trained. The old string format would append `<|end_of_text|>` instead.
3. For a PEFT model TRL now clones the loaded adapter under the name `ref` and uses that frozen copy as the reference, so the reference is the **SFT model**, as intended. The save cell writes only the `default` adapter.
4. Cached reference log-probs are only read when `trainer.precompute_ref_logps` is true. The cache cell computes them in resumable shards and then sets that flag itself.

**Reasoning strength:** DPO prompts are rendered with the same `reasoning_strength` the SFT run recorded in its sentinel. A mismatch raises.

**Run status.** Run 1 (beta 0.05, one epoch, 446 steps) completed on the GPU on 2026-11-10 and over-trained; see the next cell. **The saved cell outputs below are still run 1's**, produced by the run-1 code, and stay until the notebook is run again. The v2 settings (beta 0.25, 100-step cap, held-out evaluation, early stop, checkpoint selection) have **not been run**: the holdout split was dry-run on the host against run 1's cached dataset, and the evaluation path was read in the TRL 1.15.0 and transformers 5.17.0 sources, not executed.

## Run 1 (2026-11-10) and why the settings changed

Run 1 used lr 5e-6, cosine, 44 warmup steps, beta 0.05, sigmoid loss, batch 1 x 8, one epoch = 446 steps over all 3,564 pairs. The reference (TRL's frozen `ref` copy of the SFT adapter) was correct. The run was not. From `output/biblical_muse_glimmer_30b_unsloth_4bit_dpo/train/checkpoint-446/trainer_state.json`:

| Step | Loss | Reward accuracy | Reward, chosen | Reward, rejected | Margin |
|---|---|---|---|---|---|
| 5 | 0.688 | 0.43 | -0.001 | -0.01 | 0.01 |
| 15 | 0.485 | 1.00 | +0.010 | -0.50 | 0.51 |
| 30 | 0.171 | 1.00 | -0.016 | -2.48 | 2.47 |
| 60 | 0.003 | 1.00 | -0.33 | -10.2 | 9.8 |
| 200 | 0.001 | 1.00 | -1.11 | -16.9 | 15.8 |
| 445 | 0.002 | 1.00 | -1.38 | -17.8 | 16.4 |

Reward = beta x (policy log-prob - reference log-prob), summed over the answer's tokens.

- **Solved almost at once.** Reward accuracy is 1.00 from step 15 to the end; loss is under 0.01 by step 60. About 385 of the 446 steps trained on a finished task.
- **The weights kept moving.** The margin grew from 9.8 (step 60) to about 16 (mean over steps 300+). At beta 0.05 that is about 320 nats of shift away from the SFT model.
- **The preferred answers got less likely.** Chosen reward is positive only for steps 10-25, then falls to about -1.4: about 27 nats below the SFT adapter on answers whose reference log-prob averages -475.
- **Nothing measured or stopped it.** No held-out pairs, no early stop, and `SAVE_TOTAL_LIMIT = 3` deleted every checkpoint before step 350.

**Why the pairs are this easy** (run 1's cached reference log-probs, per token, under the SFT adapter):

| Strategy | Pairs | Chosen tokens | Rejected tokens | nats/token, chosen | nats/token, rejected | SFT already prefers chosen (per token) |
|---|---|---|---|---|---|---|
| `shallow_platitude` | 1,200 | 371 | 47 | -1.30 | -3.00 | 100.0% |
| `scripture_fabrication` | 1,200 | 368 | 355 | -1.29 | -1.89 | 92.1% |
| `voice_drift` | 1,164 | 371 | 526 | -1.31 | -1.41 | 52.9% |

The chosen answers are the SFT gold answers, and two of the three rejection styles are already far less likely under the SFT adapter than the chosen answers. DPO separates them by suppressing a style, not by learning a finer preference. That is a data problem; the settings below only limit the damage.

**What v2 changes**

| Setting | Run 1 | v2 | Why |
|---|---|---|---|
| `DPO_BETA` | 0.05 | 0.25 | The loss saturates at a reward margin of about 5; that is 100 nats of shift at 0.05 and 20 nats at 0.25. |
| Steps | 446 (one epoch) | `MAX_STEPS = 100` | The task was solved by step 60. |
| Warmup | 44 steps | 10 steps | Peak LR arrived after the task was solved. |
| Checkpoints | every 50, keep 3 | every 25, keep all | The early adapters are the candidates. |
| Held-out pairs | none | 180 (60 per strategy), evaluated every 25 steps | Measures drift on pairs the run never trains on. |
| Stop rule | none | stop when held-out chosen reward < 0 | The save cell then keeps the last evaluated step where it was still >= 0. |
| Output | `..._dpo` | `..._dpo_v2` | Run 1's adapter and checkpoints are not overwritten; its reference log-prob cache is reused read-only. |

Run 1's adapter should not be served without first comparing it against the SFT-only adapter.


## 1. Environment Preparation

Identical to the SFT notebook's environment cell. Safe to re-run.

**Do not run this cell while another kernel is training** — it upgrades packages in the shared container underneath the live run. If the SFT notebook ran in this container session and nothing was recreated since, skip this cell and start at Setup.

**Restart the kernel after this cell, then continue from Setup.**

In [1]:
import os, sys, subprocess, importlib, importlib.util

def _pip(*args, env_extra=None):
    """Run pip against this kernel's interpreter; print output only on failure."""
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(
        [sys.executable, "-m", "pip", *args], capture_output=True, text=True, env=env
    )
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# --- 1. Verify the CUDA PyTorch build is intact ------------------------------
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "cpu" in torch.__version__:
        print("  CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")
print(f"  torch {torch.__version__} - CUDA {torch.version.cuda} - GPU: {torch.cuda.get_device_name(0)}")

# --- 2. Core training packages ------------------------------------------------
print("  Installing core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...")
_pip("install", "-q", "-U", "unsloth", "trl", "accelerate", "datasets", "bitsandbytes")

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel >=0.16 on PyPI, so
# uninstall - peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
_pip("uninstall", "-y", "-q", "torchao")

# --- 3. transformers + peft from git main (only if needed) --------------------
# Muse Glimmer needs the `muse_glimmer` model package in transformers. Look for it on disk
# rather than importing transformers here (unsloth must be imported first, step 6). If it is
# already present, leave transformers and peft exactly as installed.
_tf_spec = importlib.util.find_spec("transformers")
_tf_dir = os.path.dirname(_tf_spec.origin) if _tf_spec and _tf_spec.origin else None
if _tf_dir and os.path.isdir(os.path.join(_tf_dir, "models", "muse_glimmer")):
    print("  transformers already has muse_glimmer - leaving transformers + peft as installed")
else:
    print("  transformers has no muse_glimmer - installing transformers + peft from git main...")
    _pip("install", "-q", "-U", "git+https://github.com/huggingface/transformers.git")
    _pip("install", "-q", "-U", "git+https://github.com/huggingface/peft.git")

# --- 4. Small utility packages ------------------------------------------------
for _module, _install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(_module) is None:
        print(f"  Installing {_install_args[-1]}...")
        _pip(*_install_args)

# --- 5. Fix causal_conv1d -----------------------------------------------------
# The NGC image ships the causal_conv1d Python package WITHOUT its compiled CUDA
# extension (causal_conv1d_cuda). That hard-crashes any import that reaches the
# FalconH1 model inside transformers or unsloth, so it must be fixed BEFORE
# importing either.
#
# pip also caches a broken prebuilt aarch64 wheel, so --no-binary is required to
# force a source build, together with CAUSAL_CONV1D_FORCE_BUILD=TRUE. The first
# build takes a few minutes on aarch64; pip caches the result afterwards.
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",   # DGX Spark GB10 = sm_120
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    # Clean up - don't leave partial imports that spoil unsloth's import order
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing - rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    _ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
               "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if _ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext - uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed - uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# --- 6. Import unsloth FIRST, then transformers -------------------------------
# Unsloth must be imported before transformers/trl/peft so its monkey-patches apply.
# Purge anything already loaded so the imports pick up the versions installed above.
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
import peft
import trl

print()
print(f"  unsloth:       {unsloth.__version__}")
print(f"  transformers:  {transformers.__version__}")
print(f"  peft:          {peft.__version__}")
print(f"  trl:           {trl.__version__}")
print(f"  causal_conv1d: {'OK' if _causal_ok else 'FALLBACK (torch path)'}")
print(f"  torchao:       {'PRESENT (should be absent)' if importlib.util.find_spec('torchao') else 'absent (correct)'}")
if importlib.util.find_spec("transformers.models.muse_glimmer") is None:
    raise RuntimeError(
        f"transformers {transformers.__version__} has no muse_glimmer model package - "
        "Muse Glimmer cannot be loaded. See the pip output above."
    )
print(f"  muse_glimmer:  present in transformers")
print()
print("Environment ready. Restart kernel, then rerun from Cell 1 (Configuration).")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 - CUDA 13.0 - GPU: NVIDIA GB10
  Installing core packages (unsloth, trl, accelerate, datasets, bitsandbytes)...
  transformers already has muse_glimmer - leaving transformers + peft as installed
  causal_conv1d: OK (CUDA extension loaded)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

  unsloth:       2026.11.3
  transformers:  5.17.0
  peft:          0.21.1.dev0
  trl:           1.15.0
  causal_conv1d: OK
  torchao:       absent (correct)
  muse_glimmer:  present in transformers

Environment ready. Restart kernel, then rerun from Cell 1 (Configuration).


## 2. Setup

In [2]:
import os, json
from pathlib import Path

# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and the broken
# torch.compile path (docs/dgx_spark_gb10_quirks.md section 4). Must be set BEFORE
# `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
# No PYTORCH_CUDA_ALLOC_CONF and no set_per_process_memory_fraction, per
# docs/dpo_notebook_guidelines.md ("New DPO notebooks should set neither"). The training
# cell prints allocated/reserved every CLEANUP_STEPS so a climb is visible early.

# =========================== HUGGING FACE CACHE ===========================
# Use the default Hub cache: /root/.cache/huggingface/hub in the unsloth-notebook container,
# which is bind-mounted from /home/spark/.cache/huggingface/hub on the host. Do NOT override
# HF_HUB_CACHE.

import torch
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU available. Run this in the unsloth-notebook container.")

import unsloth          # before transformers, so its patches apply
import transformers, peft, trl

if os.path.exists("/workspace/training/biblical"):
    PROJECT_ROOT = Path("/workspace/training/biblical")
elif os.path.exists("/workspace/biblical"):
    PROJECT_ROOT = Path("/workspace/biblical")
else:
    PROJECT_ROOT = Path("/home/spark/projects/training/biblical")

# Must match the base recorded by the SFT adapter at SFT_LORA_PATH (checked below).
# Meta's own repo; every from_pretrained passes use_exact_model_name=True so Unsloth's mapper
# does not swap in unsloth/Muse-Glimmer-30B-unsloth-bnb-4bit.
BASE_LLM = "meta-models/Muse-Glimmer-30B"
# Source SFT notebook: biblical_muse_glimmer_30b_sft_unsloth_4bit.ipynb
SFT_MODEL_NAME_BASE = "biblical_muse_glimmer_30b_unsloth_4bit"
# v2: run 1 (..._dpo, beta 0.05, 446 steps) over-trained; see the "Run 1" cell at the top.
# A new name keeps run 1's adapter and checkpoints, and stops the Train cell resuming from
# run 1's checkpoint-446.
MODEL_NAME_BASE = "biblical_muse_glimmer_30b_unsloth_4bit_dpo_v2"
RUN1_MODEL_NAME_BASE = "biblical_muse_glimmer_30b_unsloth_4bit_dpo"
DPO_DATA_FILE = PROJECT_ROOT / "data" / "training-data" / "biblical_persona_v2" / "biblical_personas_v2_dpo.jsonl"
SFT_LORA_PATH = PROJECT_ROOT / "output" / SFT_MODEL_NAME_BASE / "lora_adapters"
OUTPUT_BASE = PROJECT_ROOT / "output" / MODEL_NAME_BASE
TRAIN_DIR = OUTPUT_BASE / "train"
LORA_OUTPUT_DIR = OUTPUT_BASE / "lora_adapters"
# Run 1's reference log-probs (about 2 h 10 min of GPU). They depend on the SFT adapter and the
# formatted data, not on beta or step count, so the cache cell reads them from here when the
# fingerprint matches. Read-only: nothing under run 1's directory is ever written or deleted.
REF_LOGPROBS_REUSE_DIR = PROJECT_ROOT / "output" / RUN1_MODEL_NAME_BASE / "train" / "ref_logprobs_cache"

# =========================== REASONING STRENGTH ===========================
# Glimmer's template always appends "Reasoning strength: <level>." to the system prompt.
# DPO prompts must carry the SAME level the SFT run trained under, or preferences are
# scored under a system-prompt line the adapter never saw. Checked against the SFT
# sentinel below. Changing it changes both cache fingerprints, so the caches rebuild.
REASONING_STRENGTH = "low"
REASONING_MARKER = f"Reasoning strength: {REASONING_STRENGTH}."

# What every completion must look like once TRL has rendered it through the chat template:
# the generation prompt ends at '<|start|>assistant', and the completion is the channel
# header + answer + end-of-turn. This is the framing SFT trained (to=user answers closed by
# <|eot|>). The formatting cell verifies it on every row.
COMPLETION_HEAD = " to=user<|message|>"
COMPLETION_END_TOKEN = "<|eot|>"

# =========================== DPO HYPERPARAMETERS ===========================
# MAX_SEQ_LENGTH: measured on 2026-11-09 with Meta's tokenizer and template at
# reasoning_strength="low": 3,564 pairs, prompt max 493 tokens, prompt + longer completion
# median 826, p99 1,293, max 1,496. 2048 fits every row; the formatting cell re-measures
# and drops (and reports) anything longer. TRL 1.x has no max_prompt_length.
MAX_SEQ_LENGTH = 2048
BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 5e-6
# Beta anchors the policy to the SFT reference. The sigmoid loss is saturated at a reward
# margin of about 5, i.e. 5 / beta nats of log-prob shift: 100 nats at 0.05 (run 1), 20 at 0.25.
DPO_BETA = 0.25
# Run 1 reached reward accuracy 1.00 at step 15 and loss < 0.01 at step 60 of 446. The step
# count is now a hard cap, not one epoch (100 steps x 8 = 800 of the training pairs).
MAX_STEPS = 100
WARMUP_STEPS = 10
# Every evaluated step must have a checkpoint, and none may be deleted: the Save cell picks one.
SAVE_STEPS = 25
SAVE_TOTAL_LIMIT = None   # keep all (about 2 GB per checkpoint)
# Held-out pairs: never trained on, evaluated every EVAL_STEPS. Training stops at the first
# evaluation where the held-out chosen reward is negative (the policy has become LESS likely
# than the SFT adapter to produce the preferred answers).
EVAL_STEPS = 25
HOLDOUT_PER_SOURCE = 60   # per rejection strategy: 180 pairs, about 7 min per evaluation
HOLDOUT_SEED = 3407
# Save cell: None = the last evaluated step whose held-out chosen reward is >= 0.
# Set to a checkpoint step (e.g. 25) to save that checkpoint regardless.
SAVE_STEP_OVERRIDE = None
CLEANUP_STEPS = 50
LOSS_TYPE = "sigmoid"
DPO_MAX_PAIRS = 0   # 0 = use all valid pairs; set e.g. 500/1000/3000 for capped runs

# Meta's recommended sampling for Muse Glimmer (model card / generation_config.json).
GEN_TEMPERATURE = 1.0
GEN_TOP_P = 0.95
GEN_TOP_K = 64
GEN_MAX_NEW_TOKENS = 512

if EVAL_STEPS % SAVE_STEPS:
    raise ValueError(f"EVAL_STEPS ({EVAL_STEPS}) must be a multiple of SAVE_STEPS ({SAVE_STEPS}): "
                     "the Save cell loads the checkpoint of an evaluated step.")

TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# ---- The SFT run must be finished, on the same base, at the same reasoning strength ----
for path, label in [(DPO_DATA_FILE, "DPO data"), (SFT_LORA_PATH, "SFT LoRA directory")]:
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
_sft_sentinel_path = SFT_LORA_PATH / "complete.json"
if not _sft_sentinel_path.exists():
    raise FileNotFoundError(
        f"{_sft_sentinel_path} is missing. The SFT notebook has not finished (it writes this "
        "file in its Save cell). Do not start DPO on a partial adapter."
    )
SFT_SENTINEL = json.load(open(_sft_sentinel_path))
_sft_adapter_cfg = json.load(open(SFT_LORA_PATH / "adapter_config.json"))
_problems = []
if SFT_SENTINEL.get("status") != "complete":
    _problems.append(f"SFT sentinel status is {SFT_SENTINEL.get('status')!r}, not 'complete'")
if SFT_SENTINEL.get("reasoning_strength") != REASONING_STRENGTH:
    _problems.append(f"SFT trained at reasoning_strength={SFT_SENTINEL.get('reasoning_strength')!r}, "
                     f"this notebook is set to {REASONING_STRENGTH!r}")
if _sft_adapter_cfg.get("base_model_name_or_path") != BASE_LLM:
    _problems.append(f"SFT adapter base is {_sft_adapter_cfg.get('base_model_name_or_path')!r}, "
                     f"BASE_LLM is {BASE_LLM!r}")
if _problems:
    raise RuntimeError("SFT adapter does not match this DPO setup:\n  - " + "\n  - ".join(_problems))

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"unsloth {unsloth.__version__} | transformers {transformers.__version__} | "
      f"peft {peft.__version__} | trl {trl.__version__}")
print(f"HF hub cache: {os.environ.get('HF_HUB_CACHE', '<default>')}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Base model:   {BASE_LLM}")
print(f"DPO data:     {DPO_DATA_FILE}")
print(f"SFT LoRA in:  {SFT_LORA_PATH}  (step {SFT_SENTINEL.get('global_step')}, "
      f"loss {SFT_SENTINEL.get('final_loss')})")
print(f"DPO LoRA out: {LORA_OUTPUT_DIR}")
print(f"Reasoning:    {REASONING_MARKER!r} (matches the SFT sentinel)")
print(f"Max length:   {MAX_SEQ_LENGTH} | batch {BATCH_SIZE} x accum {GRAD_ACCUM} | lr {LEARNING_RATE} | beta {DPO_BETA}")
print(f"Steps:        {MAX_STEPS} max, {WARMUP_STEPS} warmup")
print(f"Checkpoints:  every {SAVE_STEPS} steps, keep {'all' if SAVE_TOTAL_LIMIT is None else SAVE_TOTAL_LIMIT}")
print(f"Held-out:     {HOLDOUT_PER_SOURCE} pairs per rejection strategy, evaluated every {EVAL_STEPS} steps; "
      "stop when chosen reward < 0")
print(f"Ref logprobs: reuse {REF_LOGPROBS_REUSE_DIR} if its fingerprint matches (read-only)")

GPU: NVIDIA GB10
unsloth 2026.11.3 | transformers 5.17.0 | peft 0.21.1.dev0 | trl 1.15.0
HF hub cache: <default>
Project root: /workspace/training/biblical
Base model:   meta-models/Muse-Glimmer-30B
DPO data:     /workspace/training/biblical/data/training-data/biblical_persona_v2/biblical_personas_v2_dpo.jsonl
SFT LoRA in:  /workspace/training/biblical/output/biblical_muse_glimmer_30b_unsloth_4bit/lora_adapters  (step 544, loss 1.3044319477151423)
DPO LoRA out: /workspace/training/biblical/output/biblical_muse_glimmer_30b_unsloth_4bit_dpo_v2/lora_adapters
Reasoning:    'Reasoning strength: low.' (matches the SFT sentinel)
Max length:   2048 | batch 1 x accum 8 | lr 5e-06 | beta 0.25
Steps:        100 max, 10 warmup
Checkpoints:  every 25 steps, keep all
Held-out:     60 pairs per rejection strategy, evaluated every 25 steps; stop when chosen reward < 0
Ref logprobs: reuse /workspace/training/biblical/output/biblical_muse_glimmer_30b_unsloth_4bit_dpo/train/ref_logprobs_cache if its fing

## 3. Load Model, Attach The SFT Adapter, Format The DPO Dataset Cache

### Verifying the inherited adapter scope

This notebook does **not** call `get_peft_model`. It attaches the SFT adapter and continues training it, so it inherits that adapter's module set exactly.

The perception-encoder freeze therefore cannot be established here, only **verified**. If the SFT run leaked adapters into `meta-models/Muse-Glimmer-30B`'s vision tower, adapter or projection, this run would inherit the leak, spend hours on it, and produce an adapter vLLM refuses. The check runs right after the adapter loads, before any expensive work.

**Memory:** the load reads the full 59.6 GB of bf16 weights into host RAM and quantizes them to nf4 on the way to the GPU. Measured on 2026-11-09: about 83 GiB in use at the peak for roughly a minute, about 36 GiB once loaded. Stop serving containers first, and if the load fails with "does not fit in GPU memory" while the box is otherwise idle, the page cache is full — drop it (`sudo sh -c 'sync; echo 3 > /proc/sys/vm/drop_caches'`) and re-run this cell.

In [3]:
import hashlib, json, os, random, shutil
from collections import Counter, defaultdict
from datasets import Dataset as HFDataset, load_from_disk
from peft import PeftModel
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    use_exact_model_name=True,   # load Meta's repo as-is; no mapper swap to unsloth's bnb-4bit re-pack
)
_resolved = getattr(model.config, "_name_or_path", "")
if "unsloth" in _resolved.lower() or BASE_LLM.split("/")[-1] not in _resolved:
    raise RuntimeError(f"Expected the Meta repo, but Unsloth loaded: {_resolved!r}")

# ---- Multimodal base: unwrap the Processor (same as the SFT notebook) --------------
# A *ForConditionalGeneration checkpoint comes back with a Processor. TRL decides between
# its text and vision pipelines by the TYPE of processing_class, so the inner tokenizer is
# what gets passed to DPOTrainer. `processor` is kept so the saved adapter directory has
# the processor files too.
processor = None
if hasattr(tokenizer, "tokenizer"):
    processor = tokenizer
    tokenizer = processor.tokenizer
    print("  (Extracted tokenizer from Processor - text-only DPO mode)")
if not getattr(tokenizer, "chat_template", None):
    if processor is not None and getattr(processor, "chat_template", None):
        tokenizer.chat_template = processor.chat_template
        print("  (Copied chat template from Processor to tokenizer)")
    else:
        raise RuntimeError("Neither the tokenizer nor the Processor carries a chat template.")

# Glimmer ships a dedicated pad token (<|finetune_right_pad|>); the fallback is defensive.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

# Attach the saved SFT adapter directly, trainable. This bypasses Unsloth's adapter
# auto-reload path and keeps the output a single LoRA against the base model.
model = PeftModel.from_pretrained(model, str(SFT_LORA_PATH), is_trainable=True)
print(f"Loaded {BASE_LLM} (resolved: {_resolved}) + SFT adapter from {SFT_LORA_PATH}")
print(f"  BOS / EOS / pad: {tokenizer.bos_token!r} / {tokenizer.eos_token!r} / {tokenizer.pad_token!r}")

# ============ VERIFY THE INHERITED ADAPTER SCOPE AND THE TOWER FREEZE ============
# Same two checks as the SFT notebook (docs/multimodal_and_hybrid_base_models.md), run
# BEFORE the trainer exists: DPOTrainer adds a second, frozen adapter copy named "ref".
def _zone_of(param_name):
    """Return the non-language sub-model a parameter belongs to, or None."""
    n = param_name.replace("base_model.model.", "", 1)
    if "vision_tower" in n or ".visual." in n or n.startswith("visual."):
        return "vision tower"
    if "vision_adapter" in n or "vision_projection" in n or "multi_modal_projector" in n:
        return "vision projector"
    if "audio_tower" in n:
        return "audio tower"
    if n.startswith("mtp.") or ".mtp." in n:
        return "mtp head"
    return None


_adapted = sorted({
    n.split(".lora_A")[0].split(".lora_B")[0].replace("base_model.model.", "")
    for n, _ in model.named_parameters() if ".lora_A." in n or ".lora_B." in n
})
_outside = [n for n in _adapted if _zone_of(n) or "language_model" not in n]
if not _adapted or _outside:
    raise RuntimeError(
        f"LoRA scope is wrong: {len(_adapted)} module(s) adapted, {len(_outside)} of them outside "
        f"model.language_model (e.g. {_outside[:3]}). This scope is INHERITED from the SFT adapter "
        "and cannot be fixed here. Re-run SFT. Do not train - vLLM refuses adapters carrying "
        "vision-layer LoRA."
    )
_unfrozen, _zone_params = {}, Counter()
for _n, _p in model.named_parameters():
    _z = _zone_of(_n)
    if _z is None:
        continue
    _zone_params[_z] += 1
    if _p.requires_grad:
        _unfrozen.setdefault(_z, []).append(_n)
_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
if _trainable == 0:
    raise RuntimeError("The SFT adapter loaded with no trainable parameters (is_trainable did not apply).")
print(f"  LoRA adapted {len(_adapted)} modules, all inside the language model "
      f"(SFT sentinel recorded {SFT_SENTINEL.get('adapted_modules')}). Trainable params: {_trainable:,}")
if SFT_SENTINEL.get("adapted_modules") not in (None, len(_adapted)):
    raise RuntimeError(f"Adapter has {len(_adapted)} adapted modules but the SFT sentinel recorded "
                       f"{SFT_SENTINEL.get('adapted_modules')}. Wrong or modified adapter directory?")
for _z, _c in sorted(_zone_params.items()):
    _bad = len(_unfrozen.get(_z, []))
    print(f"  {_z:<16} {_c:>5} params  {'FROZEN' if not _bad else str(_bad) + ' TRAINABLE - BAD'}")
if _unfrozen:
    raise RuntimeError(
        f"{sum(len(v) for v in _unfrozen.values())} vision/projector parameter(s) are trainable "
        f"(e.g. {[n for v in _unfrozen.values() for n in v][:3]}). Text-only fine-tune - they must stay frozen."
    )

# ============================ FORMATTED DATASET CACHE ============================
# TRL 1.x conversational format: `prompt` = [system, user] messages, `chosen` / `rejected` =
# [assistant message]. TRL tokenizes prompt (with add_generation_prompt=True) and
# prompt+completion through the chat template, passing each row's `chat_template_kwargs`,
# and takes the completion as everything after their common prefix.
CHAT_TEMPLATE_KWARGS = {"reasoning_strength": REASONING_STRENGTH}


def _render_ids(messages, **kw):
    return tokenizer.apply_chat_template(messages, tokenize=True, return_dict=True,
                                         **CHAT_TEMPLATE_KWARGS, **kw)["input_ids"]


def _common_prefix(a, b):
    n = 0
    for x, y in zip(a, b):
        if x != y:
            break
        n += 1
    return n


FORMATTED_CACHE_DIR = TRAIN_DIR / "formatted_dpo_dataset_cache"
FORMATTED_DATASET_DIR = FORMATTED_CACHE_DIR / "dataset"
FORMATTED_MANIFEST = FORMATTED_CACHE_DIR / "manifest.json"
FORMATTED_CACHE_DIR.mkdir(parents=True, exist_ok=True)

format_config = {
    "cache_version": 1,
    "format": "trl-conversational",
    "dpo_data_file": str(DPO_DATA_FILE),
    "dpo_data_mtime_ns": DPO_DATA_FILE.stat().st_mtime_ns,
    "max_seq_length": MAX_SEQ_LENGTH,
    "dpo_max_pairs": DPO_MAX_PAIRS,
    "reasoning_strength": REASONING_STRENGTH,
    "chat_template_sha256": hashlib.sha256(tokenizer.chat_template.encode("utf-8")).hexdigest(),
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_vocab_size": len(tokenizer),
}
format_fingerprint = hashlib.sha256(json.dumps(format_config, sort_keys=True).encode("utf-8")).hexdigest()
manifest = json.load(open(FORMATTED_MANIFEST)) if FORMATTED_MANIFEST.exists() else None

if manifest and manifest.get("fingerprint") == format_fingerprint and FORMATTED_DATASET_DIR.exists():
    dpo_dataset = load_from_disk(str(FORMATTED_DATASET_DIR))
    print(f"Loaded formatted dataset cache: {FORMATTED_DATASET_DIR}")
else:
    with open(DPO_DATA_FILE) as f:
        raw_pairs = [json.loads(line) for line in f]
    print(f"Loaded {len(raw_pairs):,} DPO pairs")
    for source, count in Counter(p.get("source", "unknown") for p in raw_pairs).most_common():
        print(f"  {source:<24} {count:>5}")

    _eot_id = tokenizer.convert_tokens_to_ids(COMPLETION_END_TOKEN)
    _head_ids = tokenizer(text=COMPLETION_HEAD, add_special_tokens=False)["input_ids"]
    formatted_pairs, errors, framing_errors = [], [], []
    skipped_too_long, max_total, max_prompt = 0, 0, 0
    for index, pair in enumerate(raw_pairs):
        chosen_msgs = pair.get("chosen", [])
        rejected_msgs = pair.get("rejected", [])
        if len(chosen_msgs) != 3 or len(rejected_msgs) != 3:
            errors.append(f"pair {index}: expected three chosen/rejected messages")
            continue
        if chosen_msgs[:-1] != rejected_msgs[:-1]:
            errors.append(f"pair {index}: prompt messages differ")
            continue
        if [m["role"] for m in chosen_msgs] != ["system", "user", "assistant"] or rejected_msgs[-1]["role"] != "assistant":
            errors.append(f"pair {index}: roles are not system/user/assistant")
            continue
        if not chosen_msgs[-1]["content"].strip() or not rejected_msgs[-1]["content"].strip():
            errors.append(f"pair {index}: empty chosen or rejected answer")
            continue
        prompt_msgs = chosen_msgs[:-1]
        # Exactly what DPOTrainer._prepare_dataset does, so the checks see what training sees.
        prompt_ids = _render_ids(prompt_msgs, add_generation_prompt=True)
        chosen_ids = _render_ids(chosen_msgs)
        rejected_ids = _render_ids(rejected_msgs)
        prompt_len = min(_common_prefix(prompt_ids, chosen_ids), _common_prefix(prompt_ids, rejected_ids))
        c_ids, r_ids = chosen_ids[prompt_len:], rejected_ids[prompt_len:]
        if (prompt_len != len(prompt_ids)
                or c_ids[:len(_head_ids)] != _head_ids or r_ids[:len(_head_ids)] != _head_ids
                or c_ids[-1] != _eot_id or r_ids[-1] != _eot_id
                or prompt_ids.count(tokenizer.bos_token_id) != 1):
            framing_errors.append(index)
            continue
        total = prompt_len + max(len(c_ids), len(r_ids))
        if total > MAX_SEQ_LENGTH:
            skipped_too_long += 1
            continue
        max_total, max_prompt = max(max_total, total), max(max_prompt, prompt_len)
        formatted_pairs.append({
            "prompt": prompt_msgs,
            "chosen": [chosen_msgs[-1]],
            "rejected": [rejected_msgs[-1]],
            "chat_template_kwargs": dict(CHAT_TEMPLATE_KWARGS),
            "source": pair.get("source", "unknown"),
            "persona": pair.get("persona", "unknown"),
        })

    if framing_errors:
        raise RuntimeError(
            f"{len(framing_errors)} pair(s) (first: {framing_errors[:5]}) do not render as "
            f"'<|start|>assistant' + {COMPLETION_HEAD!r} + answer + {COMPLETION_END_TOKEN!r} with a single BOS. "
            "The chat template or tokenizer is not the one this notebook was verified against; "
            "do not train on a framing SFT never saw."
        )
    if errors:
        print(f"Validation errors: {len(errors)}")
        for error in errors[:10]:
            print(" ", error)
    if not formatted_pairs:
        raise RuntimeError("No DPO pairs remain after validation/filtering.")

    if DPO_MAX_PAIRS and len(formatted_pairs) > DPO_MAX_PAIRS:
        random.seed(3407)
        by_source = defaultdict(list)
        for pair in formatted_pairs:
            by_source[pair["source"]].append(pair)
        sampled, total_pairs = [], len(formatted_pairs)
        for items in by_source.values():
            sample_count = max(1, round(DPO_MAX_PAIRS * len(items) / total_pairs))
            sampled.extend(random.sample(items, min(sample_count, len(items))))
        random.shuffle(sampled)
        formatted_pairs = sampled[:DPO_MAX_PAIRS]
        print(f"Capped formatted DPO dataset to {len(formatted_pairs):,} pairs")

    dpo_dataset = HFDataset.from_list(formatted_pairs).shuffle(seed=3407)
    if FORMATTED_DATASET_DIR.exists():
        shutil.rmtree(FORMATTED_DATASET_DIR)
    dpo_dataset.save_to_disk(str(FORMATTED_DATASET_DIR))
    tmp_manifest = FORMATTED_MANIFEST.with_suffix(".json.tmp")
    with open(tmp_manifest, "w") as f:
        json.dump({"status": "complete", "fingerprint": format_fingerprint, "config": format_config,
                   "rows": len(dpo_dataset), "max_total_tokens": max_total, "max_prompt_tokens": max_prompt}, f, indent=2)
    os.replace(tmp_manifest, FORMATTED_MANIFEST)
    print(f"Saved formatted dataset cache: {FORMATTED_DATASET_DIR}")
    print(f"  Longest prompt: {max_prompt} tokens | longest prompt+completion: {max_total} tokens (limit {MAX_SEQ_LENGTH})")
    print(f"  Dropped for length: {skipped_too_long:,} | structural errors: {len(errors):,}")
    print(f"  Framing: every completion is {COMPLETION_HEAD!r} + answer + {COMPLETION_END_TOKEN!r}")

print(f"DPO rows: {len(dpo_dataset):,}")

==((====))==  Unsloth 2026.11.3: Fast Muse_Glimmer patching. Transformers: 5.17.0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.688 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/1436 [00:00<?, ?it/s]

  (Extracted tokenizer from Processor - text-only DPO mode)
Loaded meta-models/Muse-Glimmer-30B (resolved: meta-models/Muse-Glimmer-30B) + SFT adapter from /workspace/training/biblical/output/biblical_muse_glimmer_30b_unsloth_4bit/lora_adapters
  BOS / EOS / pad: '<|begin_of_text|>' / '<|end_of_text|>' / '<|finetune_right_pad|>'
  LoRA adapted 416 modules, all inside the language model (SFT sentinel recorded 416). Trainable params: 209,584,128
  vision projector     3 params  FROZEN
  vision tower       806 params  FROZEN
Loaded 3,564 DPO pairs
  shallow_platitude         1200
  scripture_fabrication     1200
  voice_drift               1164


Saving the dataset (0/1 shards):   0%|          | 0/3564 [00:00<?, ? examples/s]

Saved formatted dataset cache: /workspace/training/biblical/output/biblical_muse_glimmer_30b_unsloth_4bit_dpo_v2/train/formatted_dpo_dataset_cache/dataset
  Longest prompt: 493 tokens | longest prompt+completion: 1496 tokens (limit 2048)
  Dropped for length: 0 | structural errors: 0
  Framing: every completion is ' to=user<|message|>' + answer + '<|eot|>'
DPO rows: 3,564


## 4. Configure DPO Trainer

In [4]:
import inspect, math, torch
from trl import DPOTrainer, DPOConfig

# This notebook is written against the TRL 1.x DPO trainer. Fail here, with a reason,
# rather than hours later inside the cache or the loss.
_ref_sig = list(inspect.signature(DPOTrainer.compute_ref_log_probs).parameters)
if _ref_sig[:3] != ["self", "model", "inputs"]:
    raise RuntimeError(
        f"TRL {trl.__version__}: DPOTrainer.compute_ref_log_probs has signature {_ref_sig}, not "
        "(self, model, inputs). This notebook targets TRL 1.x (verified on 1.15.0)."
    )

FastLanguageModel.for_training(model)
effective_batch = BATCH_SIZE * GRAD_ACCUM
# A hard step cap, not one epoch: run 1 had solved the objective by step 60 of 446.
max_steps = MAX_STEPS
warmup_steps = WARMUP_STEPS

trainer = DPOTrainer(
    model=model,
    ref_model=None,            # PEFT: TRL clones the loaded SFT adapter as a frozen "ref" adapter
    args=DPOConfig(
        beta=DPO_BETA,
        loss_type=LOSS_TYPE,
        max_length=MAX_SEQ_LENGTH,
        precompute_ref_log_probs=False,   # the persistent cache cell below handles this
        per_device_train_batch_size=BATCH_SIZE,
        # Held-out evaluation (Train cell). The Trainer default is 8; rows here run to ~1,500
        # tokens and each forward carries chosen + rejected.
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="cosine",
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=3407,
        gradient_checkpointing=True,
        dataloader_pin_memory=False,
        output_dir=str(TRAIN_DIR),
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        logging_steps=5,
        report_to="none",
        dataset_num_proc=1,
    ),
    # All pairs go in here so the reference-logprob cache covers every row and run 1's cache
    # can be reused. The cache cell then moves the held-out slice to trainer.eval_dataset.
    train_dataset=dpo_dataset,
    processing_class=tokenizer,    # a tokenizer, not the Processor: selects TRL's text pipeline
)

# ---- Assert the trainer took the path this notebook assumes -------------------------
if getattr(trainer, "_is_vlm", False) or getattr(trainer, "_is_vision_dataset", False):
    raise RuntimeError("DPOTrainer chose its vision pipeline. processing_class must be the inner tokenizer.")
_needed = {"prompt_ids", "chosen_ids", "rejected_ids"}
if not _needed.issubset(trainer.train_dataset.column_names):
    raise RuntimeError(f"Tokenized dataset lacks {_needed - set(trainer.train_dataset.column_names)}.")
if len(trainer.train_dataset) != len(dpo_dataset):
    raise RuntimeError(f"TRL dropped rows while tokenizing: {len(dpo_dataset)} -> {len(trainer.train_dataset)}.")

# What TRL actually tokenized, checked on real rows: completion framing and length.
_eot_id = tokenizer.convert_tokens_to_ids(COMPLETION_END_TOKEN)
_head_ids = tokenizer(text=COMPLETION_HEAD, add_special_tokens=False)["input_ids"]
_bad, _longest = 0, 0
for _row in trainer.train_dataset.select(range(min(200, len(trainer.train_dataset)))):
    for _k in ("chosen_ids", "rejected_ids"):
        if _row[_k][:len(_head_ids)] != _head_ids or _row[_k][-1] != _eot_id:
            _bad += 1
    _longest = max(_longest, len(_row["prompt_ids"]) + max(len(_row["chosen_ids"]), len(_row["rejected_ids"])))
if _bad:
    raise RuntimeError(f"{_bad} tokenized completion(s) in the first 200 rows are not "
                       f"{COMPLETION_HEAD!r} ... {COMPLETION_END_TOKEN!r}. Do not train.")
_r0 = trainer.train_dataset[0]
if REASONING_MARKER not in tokenizer.decode(_r0["prompt_ids"]):
    raise RuntimeError(f"Tokenized prompt lacks {REASONING_MARKER!r}: chat_template_kwargs did not reach the template.")

# The reference is the frozen "ref" copy of the SFT adapter; only "default" may train.
_adapters = list(model.peft_config)
_ref_trainable = [n for n, p in model.named_parameters() if ".ref." in n and p.requires_grad]
_default_trainable = sum(1 for n, p in model.named_parameters() if ".default." in n and p.requires_grad)
if "ref" not in _adapters:
    raise RuntimeError(f"DPOTrainer did not create the 'ref' adapter (adapters: {_adapters}). With no copy of "
                       "the SFT adapter the reference would be the bare base model, which is not the intent here.")
if _ref_trainable or not _default_trainable:
    raise RuntimeError(f"Adapter trainability is wrong: {len(_ref_trainable)} 'ref' tensors trainable, "
                       f"{_default_trainable} 'default' tensors trainable.")

print("DPO trainer configured.")
print(f"  TRL {trl.__version__} | text pipeline | adapters: {_adapters} (active: {model.active_adapters})")
print(f"  DPO pairs: {len(trainer.train_dataset):,} (before the held-out split) | effective batch {effective_batch} | max steps {max_steps} | warmup {warmup_steps} | beta {trainer.beta}")
print(f"  max_length: {trainer.args.max_length} | longest row in first 200: {_longest} tokens")
print(f"  prompt tail:     {tokenizer.decode(_r0['prompt_ids'][-6:])!r}")
print(f"  completion head: {tokenizer.decode(_r0['chosen_ids'][:6])!r} ... {tokenizer.decode(_r0['chosen_ids'][-2:])!r}")
print("  precompute_ref_log_probs: False at construction; the next cell builds the persistent cache")

Tokenizing train dataset:   0%|          | 0/3564 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/3564 [00:00<?, ? examples/s]

DPO trainer configured.
  TRL 1.15.0 | text pipeline | adapters: ['default', 'ref'] (active: ['default'])
  DPO pairs: 3,564 (before the held-out split) | effective batch 8 | max steps 100 | warmup 10 | beta 0.25
  max_length: 2048 | longest row in first 200: 1418 tokens
  prompt tail:     ' unfolding plan?<|eot|><|start|>assistant'
  completion head: ' to=user<|message|>I lifted my' ... '.<|eot|>'
  precompute_ref_log_probs: False at construction; the next cell builds the persistent cache


## 5. Precompute Reference Log Probabilities (Persistent, Resumable Cache)

DPO needs the frozen reference model's log probability of every chosen and rejected completion. TRL's own `precompute_ref_log_probs=True` is one-shot and in-memory, so a kernel crash during it starts over from row 0.

This cell computes them in shards under `TRAIN_DIR/ref_logprobs_cache/shards/`, writes a fingerprinted manifest, resumes from existing shards, attaches `ref_chosen_logps` / `ref_rejected_logps` to the trainer dataset, and then sets `trainer.precompute_ref_logps = True`. That last step matters on TRL 1.x: without it the loss ignores the cached columns and re-runs the reference forward pass on every step.

The reference is the frozen `ref` copy of the SFT adapter that `DPOTrainer` created, so the fingerprint includes the SFT adapter file.

**Reuse.** The fingerprint does not depend on beta, step count or output name, so run 1's finished cache (`REF_LOGPROBS_REUSE_DIR`) is loaded as-is when its fingerprint matches and every row is token-identical. It is only read; if it does not match, a new cache is built under this run's own `TRAIN_DIR` (about 2 h 10 min).

**Held-out split.** After the log-probs are attached, the cell moves `HOLDOUT_PER_SOURCE` pairs per rejection strategy to `trainer.eval_dataset` and drops any training pair that shares its prompt and chosen answer with a held-out pair. On the current data that is 180 held-out, 3,352 training, 32 dropped.

In [5]:
import hashlib, json, os, random, time, shutil
from collections import Counter, defaultdict
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_from_disk

# Re-running this cell: start again from the full tokenized set, not from the already-split one.
if getattr(trainer, "_dpo_full_dataset", None) is not None:
    trainer.train_dataset = trainer._dpo_full_dataset

REF_LOGPROBS_CACHE_DIR = TRAIN_DIR / "ref_logprobs_cache"
REF_LOGPROBS_SHARD_DIR = REF_LOGPROBS_CACHE_DIR / "shards"
REF_LOGPROBS_DATASET_DIR = REF_LOGPROBS_CACHE_DIR / "dataset"
REF_LOGPROBS_MANIFEST = REF_LOGPROBS_CACHE_DIR / "manifest.json"
REF_LOGPROBS_SHARD_SIZE = 64
REF_LOGPROBS_BATCH_SIZE = 1

REF_LOGPROBS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REF_LOGPROBS_SHARD_DIR.mkdir(parents=True, exist_ok=True)

_sft_weights = SFT_LORA_PATH / "adapter_model.safetensors"
_ref_cache_config = {
    "cache_version": 1,
    "trl_version": trl.__version__,
    "base_llm": BASE_LLM,
    "sft_lora_path": str(SFT_LORA_PATH),
    "sft_adapter_size": _sft_weights.stat().st_size,
    "sft_adapter_mtime_ns": _sft_weights.stat().st_mtime_ns,
    "reference": "ref adapter (copy of SFT)" if "ref" in model.peft_config else "base model",
    "formatted_dataset_fingerprint": format_fingerprint,
    "dataset_len": len(trainer.train_dataset),
    "max_seq_length": MAX_SEQ_LENGTH,
    "reasoning_strength": REASONING_STRENGTH,
    "shard_size": REF_LOGPROBS_SHARD_SIZE,
    "batch_size": REF_LOGPROBS_BATCH_SIZE,
}
_ref_cache_fingerprint = hashlib.sha256(
    json.dumps(_ref_cache_config, sort_keys=True).encode("utf-8")
).hexdigest()
_required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}


def _read_ref_manifest():
    if not REF_LOGPROBS_MANIFEST.exists():
        return None
    with open(REF_LOGPROBS_MANIFEST) as f:
        return json.load(f)


def _load_valid_ref_shard(shard_path, shard_idx, start, end):
    try:
        shard_payload = torch.load(shard_path, map_location="cpu")
    except Exception as exc:
        raise RuntimeError(f"Unreadable ref logprob shard: {shard_path}") from exc
    if not isinstance(shard_payload, dict):
        raise RuntimeError(f"Invalid ref logprob shard payload: {shard_path}")
    if shard_payload.get("fingerprint") != _ref_cache_fingerprint:
        raise RuntimeError(f"Stale ref logprob shard fingerprint: {shard_path}")
    if shard_payload.get("shard_idx") != shard_idx or shard_payload.get("start") != start or shard_payload.get("end") != end:
        raise RuntimeError(f"Ref logprob shard index/range mismatch: {shard_path}")
    for column in ("ref_chosen_logps", "ref_rejected_logps"):
        values = shard_payload.get(column)
        if not isinstance(values, torch.Tensor) or values.numel() != end - start or not torch.isfinite(values).all():
            raise RuntimeError(f"Invalid {column} values in ref logprob shard: {shard_path}")
    return shard_payload


def _write_ref_manifest(status, completed_shards):
    payload = {
        "status": status,
        "fingerprint": _ref_cache_fingerprint,
        "config": _ref_cache_config,
        "completed_shards": completed_shards,
        "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    tmp_path = REF_LOGPROBS_MANIFEST.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(payload, f, indent=2)
    os.replace(tmp_path, REF_LOGPROBS_MANIFEST)


# ---- Run 1's cache, read-only -------------------------------------------------------
# Same SFT adapter, same formatted data, same TRL: the fingerprint is identical, so the
# reference log-probs are too. If anything differs this is skipped and a new cache is built
# under THIS run's TRAIN_DIR. Nothing below writes to or deletes from REF_LOGPROBS_REUSE_DIR.
# Remembered on the trainer so a re-run of this cell still reports where the values came from.
_ref_cache_source = getattr(trainer, "_dpo_ref_cache_source", str(REF_LOGPROBS_CACHE_DIR))
_reuse_manifest_path = REF_LOGPROBS_REUSE_DIR / "manifest.json"
_reuse_dataset_dir = REF_LOGPROBS_REUSE_DIR / "dataset"
if (not _required_ref_cols.issubset(trainer.train_dataset.column_names)
        and REF_LOGPROBS_REUSE_DIR.resolve() != REF_LOGPROBS_CACHE_DIR.resolve()):
    _reuse_manifest = json.load(open(_reuse_manifest_path)) if _reuse_manifest_path.exists() else None
    if (_reuse_manifest is not None and _reuse_manifest.get("fingerprint") == _ref_cache_fingerprint
            and _reuse_manifest.get("status") == "complete" and _reuse_dataset_dir.exists()):
        _reuse_dataset = load_from_disk(str(_reuse_dataset_dir))
        _same_rows = len(_reuse_dataset) == len(trainer.train_dataset) and _required_ref_cols.issubset(_reuse_dataset.column_names)
        if _same_rows:
            # Row order must be identical or every log-prob would be attached to the wrong pair.
            _id_cols = ["prompt_ids", "chosen_ids", "rejected_ids"]
            _a = trainer.train_dataset.select_columns(_id_cols)
            _b = _reuse_dataset.select_columns(_id_cols)
            _same_rows = all(_ra == _rb for _ra, _rb in zip(_a, _b))
        if _same_rows:
            trainer.train_dataset = _reuse_dataset
            _ref_cache_source = trainer._dpo_ref_cache_source = str(REF_LOGPROBS_REUSE_DIR)
            print(f"Loaded run 1's ref logprob cache (read-only, all {len(_reuse_dataset):,} rows token-identical): {_reuse_dataset_dir}")
        else:
            print("Run 1's ref logprob cache has the right fingerprint but different rows; not using it.")
    else:
        print(f"Run 1's ref logprob cache is missing or has a different fingerprint; building a new one under {REF_LOGPROBS_CACHE_DIR}.")

_manifest = _read_ref_manifest()
_cache_matches = _manifest is not None and _manifest.get("fingerprint") == _ref_cache_fingerprint

if _cache_matches and _manifest.get("status") == "complete" and REF_LOGPROBS_DATASET_DIR.exists():
    cached_dataset = load_from_disk(str(REF_LOGPROBS_DATASET_DIR))
    if len(cached_dataset) == len(trainer.train_dataset) and _required_ref_cols.issubset(cached_dataset.column_names):
        trainer.train_dataset = cached_dataset
        print(f"Loaded persistent ref logprob cache: {REF_LOGPROBS_DATASET_DIR}")
    else:
        print("Ignoring stale ref logprob dataset cache: length or columns do not match")
        _cache_matches = False

if not _required_ref_cols.issubset(trainer.train_dataset.column_names):
    if not _cache_matches:
        for stale_shard in REF_LOGPROBS_SHARD_DIR.glob("shard-*.pt"):
            stale_shard.unlink()
        _write_ref_manifest("in_progress", [])

    num_rows = len(trainer.train_dataset)
    shard_ranges = [
        (start, min(start + REF_LOGPROBS_SHARD_SIZE, num_rows))
        for start in range(0, num_rows, REF_LOGPROBS_SHARD_SIZE)
    ]
    print("Persistent reference logprob cache")
    print(f"  Rows: {num_rows} | shards: {len(shard_ranges)} x {REF_LOGPROBS_SHARD_SIZE} | dir: {REF_LOGPROBS_CACHE_DIR}")
    print(f"  Reference: {_ref_cache_config['reference']}")

    # Only the columns the collator reads; the message-list columns stay out of the loader.
    _ref_source = trainer.train_dataset.select_columns(["prompt_ids", "chosen_ids", "rejected_ids"])
    completed = []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if shard_path.exists():
            try:
                _load_valid_ref_shard(shard_path, shard_idx, start, end)
            except RuntimeError as exc:
                print(f"Recomputing ref logprob shard {shard_idx}: {exc}")
                shard_path.unlink()
            else:
                completed.append(shard_idx)
                continue

        shard_loader = DataLoader(
            _ref_source.select(range(start, end)),
            batch_size=REF_LOGPROBS_BATCH_SIZE,
            collate_fn=trainer.data_collator,
            num_workers=0,
            pin_memory=False,
            shuffle=False,
        )
        shard_loader = trainer.accelerator.prepare(shard_loader)

        ref_chosen_logps, ref_rejected_logps = [], []
        for padded_batch in tqdm(shard_loader, desc=f"Ref logprobs shard {shard_idx + 1}/{len(shard_ranges)}"):
            # TRL 1.x signature: (model, inputs). With a PEFT model and no ref_model it
            # switches to the frozen "ref" adapter for this forward pass.
            ref_chosen_logp, ref_rejected_logp = trainer.compute_ref_log_probs(trainer.model, padded_batch)
            ref_chosen_logps.append(ref_chosen_logp.detach().float().cpu())
            ref_rejected_logps.append(ref_rejected_logp.detach().float().cpu())
        torch.cuda.empty_cache()

        shard_payload = {
            "fingerprint": _ref_cache_fingerprint,
            "shard_idx": shard_idx,
            "start": start,
            "end": end,
            "ref_chosen_logps": torch.cat(ref_chosen_logps),
            "ref_rejected_logps": torch.cat(ref_rejected_logps),
        }
        if not (torch.isfinite(shard_payload["ref_chosen_logps"]).all() and torch.isfinite(shard_payload["ref_rejected_logps"]).all()):
            raise RuntimeError(f"Non-finite reference log-probs in shard {shard_idx}. Stop and inspect before training.")
        tmp_shard_path = shard_path.with_suffix(".pt.tmp")
        torch.save(shard_payload, tmp_shard_path)
        os.replace(tmp_shard_path, shard_path)
        completed.append(shard_idx)
        _write_ref_manifest("in_progress", completed)

    all_ref_chosen_logps, all_ref_rejected_logps = [], []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_payload = _load_valid_ref_shard(REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt", shard_idx, start, end)
        all_ref_chosen_logps.append(shard_payload["ref_chosen_logps"])
        all_ref_rejected_logps.append(shard_payload["ref_rejected_logps"])
    ref_chosen_values = torch.cat(all_ref_chosen_logps).tolist()
    ref_rejected_values = torch.cat(all_ref_rejected_logps).tolist()
    if len(ref_chosen_values) != num_rows or len(ref_rejected_values) != num_rows:
        raise RuntimeError("Ref logprob cache length does not match training dataset")

    train_dataset_with_ref = trainer.train_dataset
    for ref_col in ("ref_chosen_logps", "ref_rejected_logps"):
        if ref_col in train_dataset_with_ref.column_names:
            train_dataset_with_ref = train_dataset_with_ref.remove_columns(ref_col)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_chosen_logps", ref_chosen_values)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_rejected_logps", ref_rejected_values)

    if REF_LOGPROBS_DATASET_DIR.exists():
        shutil.rmtree(REF_LOGPROBS_DATASET_DIR)
    train_dataset_with_ref.save_to_disk(str(REF_LOGPROBS_DATASET_DIR))
    trainer.train_dataset = train_dataset_with_ref
    _write_ref_manifest("complete", list(range(len(shard_ranges))))
    print(f"Saved persistent ref logprob dataset cache: {REF_LOGPROBS_DATASET_DIR}")

# TRL 1.x reads the cached columns ONLY when this flag is set (DPOTrainer._compute_loss).
# Without it, every training step would re-run the reference forward pass.
trainer.precompute_ref_logps = True

_rc = torch.tensor(trainer.train_dataset["ref_chosen_logps"])
_rr = torch.tensor(trainer.train_dataset["ref_rejected_logps"])
print(f"Ref logprob columns ready: {sorted(_required_ref_cols)}; trainer.precompute_ref_logps = True")
print(f"  ref chosen   mean {_rc.mean():.1f} (min {_rc.min():.1f}, max {_rc.max():.1f})")
print(f"  ref rejected mean {_rr.mean():.1f} (min {_rr.min():.1f}, max {_rr.max():.1f})")
print(f"  SFT reference already prefers chosen on {(_rc > _rr).float().mean() * 100:.1f}% of pairs "
      "(raw summed log-probs; longer answers score lower, so this is a sanity figure, not accuracy)")

# ============================ HELD-OUT SLICE ============================
# Split AFTER the reference log-probs are attached, so the cache above always covers all rows.
# Stratified by rejection strategy, fixed seed. The same question + gold answer can appear
# under more than one strategy (the datagen draws three independent samples), so any training
# row that shares its prompt and chosen answer with a held-out row is dropped too. Otherwise
# the held-out chosen reward would be measured on answers the run trains on.
_full = trainer.train_dataset
trainer._dpo_full_dataset = _full
_rng = random.Random(HOLDOUT_SEED)
_by_source = defaultdict(list)
for _i, _s in enumerate(_full["source"]):
    _by_source[_s].append(_i)
_holdout_idx = sorted(_i for _s in sorted(_by_source)
                      for _i in _rng.sample(_by_source[_s], min(HOLDOUT_PER_SOURCE, len(_by_source[_s]))))
_keys = [hashlib.sha256(json.dumps([_r["prompt_ids"], _r["chosen_ids"]]).encode("utf-8")).hexdigest()
         for _r in _full.select_columns(["prompt_ids", "chosen_ids"])]
_holdout_set = set(_holdout_idx)
_holdout_keys = {_keys[_i] for _i in _holdout_idx}
_train_idx = [_i for _i in range(len(_full)) if _i not in _holdout_set and _keys[_i] not in _holdout_keys]
HOLDOUT_LEAK_DROPPED = len(_full) - len(_holdout_idx) - len(_train_idx)
if not _holdout_idx or not _train_idx:
    raise RuntimeError("The held-out split left an empty training or held-out set.")

trainer.eval_dataset = _full.select(_holdout_idx)
trainer.train_dataset = _full.select(_train_idx)
if _holdout_keys & {_keys[_i] for _i in _train_idx}:
    raise RuntimeError("A held-out prompt + chosen answer is still present in the training set.")

_he = trainer.eval_dataset
_hc, _hr = torch.tensor(_he["ref_chosen_logps"]), torch.tensor(_he["ref_rejected_logps"])
print(f"Held-out split (seed {HOLDOUT_SEED}): {len(trainer.train_dataset):,} training pairs, {len(_he):,} held-out, "
      f"{HOLDOUT_LEAK_DROPPED} training pairs dropped for sharing a prompt + chosen answer with a held-out pair")
for _s, _n in sorted(Counter(_he["source"]).items()):
    print(f"  held-out {_s:<24} {_n:>4}   training {Counter(trainer.train_dataset['source'])[_s]:>5}")
print(f"  held-out ref chosen mean {_hc.mean():.1f}, ref rejected mean {_hr.mean():.1f}")
print(f"  {MAX_STEPS} steps x {BATCH_SIZE * GRAD_ACCUM} = {MAX_STEPS * BATCH_SIZE * GRAD_ACCUM:,} training pairs seen at most")

Loaded run 1's ref logprob cache (read-only, all 3,564 rows token-identical): /workspace/training/biblical/output/biblical_muse_glimmer_30b_unsloth_4bit_dpo/train/ref_logprobs_cache/dataset
Ref logprob columns ready: ['ref_chosen_logps', 'ref_rejected_logps']; trainer.precompute_ref_logps = True
  ref chosen   mean -474.4 (min -1331.4, max -3.9)
  ref rejected mean -508.3 (min -1629.1, max -43.6)
  SFT reference already prefers chosen on 60.3% of pairs (raw summed log-probs; longer answers score lower, so this is a sanity figure, not accuracy)
Held-out split (seed 3407): 3,352 training pairs, 180 held-out, 32 training pairs dropped for sharing a prompt + chosen answer with a held-out pair
  held-out scripture_fabrication      60   training  1132
  held-out shallow_platitude          60   training  1126
  held-out voice_drift                60   training  1094
  held-out ref chosen mean -491.6, ref rejected mean -508.4
  100 steps x 8 = 800 training pairs seen at most


## 6. Train

Run after the persistent reference-logprob cache cell. Trains for at most `MAX_STEPS`, checkpoints every `SAVE_STEPS` steps (all kept) and resumes from the newest checkpoint if one exists.

Every `EVAL_STEPS` steps the held-out pairs are scored and a `[held-out]` line is printed and appended to `TRAIN_DIR/heldout_eval.json`. **Training stops at the first evaluation where the held-out chosen reward is negative.** Each evaluation takes about 7 minutes (180 pairs, batch 1). Once the stop has fired, re-running this cell does not resume.

The memory callback runs every `CLEANUP_STEPS` steps, not every step, and prints `reserved`. That figure should plateau; a steady climb is the signal to stop.

In [6]:
import gc, json, os
from transformers import TrainerCallback
from transformers.trainer_utils import get_last_checkpoint


class PeriodicMemoryCleanup(TrainerCallback):
    """Collect dead Python objects, return cached CUDA blocks, and report memory every `every` steps."""

    def __init__(self, every=50):
        self.every = max(1, int(every))

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every == 0:
            gc.collect()
            torch.cuda.empty_cache()
            print(f"  [mem] step {state.global_step}: "
                  f"allocated {torch.cuda.memory_allocated() / 2**30:.1f} GiB, "
                  f"reserved {torch.cuda.memory_reserved() / 2**30:.1f} GiB, "
                  f"peak allocated {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
        return control


HELDOUT_LOG = TRAIN_DIR / "heldout_eval.json"


def _write_heldout_log(payload):
    tmp_path = HELDOUT_LOG.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(payload, f, indent=2)
    os.replace(tmp_path, HELDOUT_LOG)


class HeldOutRewardStop(TrainerCallback):
    """Every `every` steps, score the held-out pairs and stop once their chosen reward is negative.

    Reward = beta x (policy log-prob - SFT-reference log-prob). A negative chosen reward means the
    policy is now LESS likely than the SFT adapter to produce the preferred answers: it is buying
    margin by drifting, which is what run 1 did for 420 steps. Every evaluation is appended to
    HELDOUT_LOG so the Save cell can pick a checkpoint, also after a kernel restart.
    """

    def __init__(self, trainer, every, record):
        self.trainer, self.every, self.record = trainer, max(1, int(every)), record

    def on_step_end(self, args, state, control, **kwargs):
        step = state.global_step
        if step % self.every or any(e["step"] == step for e in self.record["evals"]):
            return control
        metrics = self.trainer.evaluate()   # trainer.eval_dataset; reads the cached ref log-probs
        # evaluate() leaves the model in eval mode, and DPOTrainer.log() files the training
        # metrics logged at this same step under whichever mode the model is in.
        self.trainer.model.train()
        missing = [k for k in ("eval_rewards/chosen", "eval_rewards/rejected", "eval_rewards/margins",
                               "eval_rewards/accuracies", "eval_loss") if k not in metrics]
        if missing:
            raise RuntimeError(f"Held-out evaluation returned no {missing}; got {sorted(metrics)}. "
                               "The stop rule cannot run - do not continue training blind.")
        chosen = float(metrics["eval_rewards/chosen"])
        entry = {
            "step": step,
            "rewards_chosen": chosen,
            "rewards_rejected": float(metrics["eval_rewards/rejected"]),
            "rewards_margins": float(metrics["eval_rewards/margins"]),
            "rewards_accuracies": float(metrics["eval_rewards/accuracies"]),
            "loss": float(metrics["eval_loss"]),
            "chosen_nats_vs_sft": chosen / self.trainer.beta,
        }
        self.record["evals"].append(entry)
        print(f"  [held-out] step {step}: chosen reward {chosen:+.4f} ({entry['chosen_nats_vs_sft']:+.1f} nats vs SFT) | "
              f"rejected {entry['rewards_rejected']:+.3f} | margin {entry['rewards_margins']:.3f} | "
              f"accuracy {entry['rewards_accuracies']:.3f} | loss {entry['loss']:.4f}")
        if chosen < 0:
            self.record["stopped_at"] = step
            control.should_training_stop = True
            print(f"  [held-out] chosen reward is negative at step {step}: stopping. "
                  "The Save cell keeps the last evaluated step where it was >= 0.")
        _write_heldout_log(self.record)
        return control


required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}
if not required_ref_cols.issubset(trainer.train_dataset.column_names) or not trainer.precompute_ref_logps:
    raise RuntimeError("Run the persistent reference-logprob cache cell before training.")
if trainer.eval_dataset is None or not required_ref_cols.issubset(trainer.eval_dataset.column_names):
    raise RuntimeError("No held-out set with reference log-probs. Run the cache cell (it makes the split) before training.")

last_checkpoint = get_last_checkpoint(trainer.args.output_dir)

# Held-out record: new for a new run; on a resume, keep only evaluations up to the checkpoint.
_resume_step = int(last_checkpoint.rsplit("-", 1)[-1]) if last_checkpoint else 0
heldout_record = {
    "beta": DPO_BETA, "eval_steps": EVAL_STEPS, "holdout_seed": HOLDOUT_SEED,
    "holdout_pairs": len(trainer.eval_dataset), "train_pairs": len(trainer.train_dataset),
    "leak_dropped": HOLDOUT_LEAK_DROPPED, "evals": [], "stopped_at": None,
}
if last_checkpoint and HELDOUT_LOG.exists():
    _previous = json.load(open(HELDOUT_LOG))
    heldout_record["evals"] = [e for e in _previous.get("evals", []) if e["step"] <= _resume_step]
    if _previous.get("stopped_at") is not None and _previous["stopped_at"] <= _resume_step:
        heldout_record["stopped_at"] = _previous["stopped_at"]
_write_heldout_log(heldout_record)

trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if not isinstance(cb, (PeriodicMemoryCleanup, HeldOutRewardStop))
    and type(cb).__name__ not in ("PeriodicMemoryCleanup", "HeldOutRewardStop")   # stale copies from a re-run of this cell
]
trainer.add_callback(PeriodicMemoryCleanup(every=CLEANUP_STEPS))
trainer.add_callback(HeldOutRewardStop(trainer, EVAL_STEPS, heldout_record))

if heldout_record["stopped_at"] is not None:
    # The stop rule already fired. Resuming would train past the point it stopped at.
    print(f"Held-out stop already fired at step {heldout_record['stopped_at']} ({HELDOUT_LOG}). "
          "Not resuming; go to the Save cell.")
else:
    print("DPO training started...")
    print("Loss starts near 0.69 (ln 2) and should fall; rewards/margins should widen (chosen > rejected).")
    print(f"Held-out pairs are scored every {EVAL_STEPS} steps; training stops when their chosen reward goes negative.\n")
    if last_checkpoint:
        print(f"Resuming from checkpoint: {last_checkpoint}")
        result = trainer.train(resume_from_checkpoint=last_checkpoint)
    else:
        result = trainer.train()

    print("\nDPO training complete")
    print(f"Final loss: {result.training_loss:.4f}")
    print(f"Total steps: {result.global_step} of {MAX_STEPS}"
          + (f" (stopped early by the held-out rule)" if heldout_record["stopped_at"] is not None else ""))

print("\nHeld-out evaluations:")
for _e in heldout_record["evals"]:
    print(f"  step {_e['step']:>4}: chosen {_e['rewards_chosen']:+.4f} ({_e['chosen_nats_vs_sft']:+.1f} nats) | "
          f"rejected {_e['rewards_rejected']:+.3f} | margin {_e['rewards_margins']:.3f} | accuracy {_e['rewards_accuracies']:.3f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 200001, 'bos_token_id': 200000}.


DPO training started...
Loss starts near 0.69 (ln 2) and should fall; rewards/margins should widen (chosen > rejected).
Held-out pairs are scored every 25 steps; training stops when their chosen reward goes negative.



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,352 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 209,584,128 of 30,195,794,944 (0.69% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logits / chosen,logits / rejected,logps / chosen,logps / rejected,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
5,0.590110,0.022644,-0.253531,0.550000,0.276175,-9.687590,-8.761089,-487.669309,-500.571187,,,,,,,,,,,,
10,0.139620,0.025465,-3.475306,1.000000,3.500771,-9.632449,-8.863321,-471.547947,-561.723915,,,,,,,,,,,,
15,0.046654,0.003422,-8.021073,1.000000,8.024495,-10.029990,-8.902349,-438.249375,-539.354952,,,,,,,,,,,,
20,0.012782,0.115552,-13.717609,1.000000,13.833161,-9.788794,-8.996609,-459.135087,-587.962172,,,,,,,,,,,,
25,0.012782,,,,,,,,,0.001252,1.388716,301546.000000,-9.675275,-9.074482,0.628708,-0.041045,-18.768594,1.000000,18.727549,-491.803618,-583.463168


  [held-out] step 25: chosen reward -0.0410 (-0.2 nats vs SFT) | rejected -18.769 | margin 18.728 | accuracy 1.000 | loss 0.0013
  [held-out] chosen reward is negative at step 25: stopping. The Save cell keeps the last evaluated step where it was >= 0.


Unsloth: Restored added_tokens_decoder metadata in /workspace/training/biblical/output/biblical_muse_glimmer_30b_unsloth_4bit_dpo_v2/train/checkpoint-25/tokenizer_config.json.



DPO training complete
Final loss: 0.1589
Total steps: 25 of 100 (stopped early by the held-out rule)

Held-out evaluations:
  step   25: chosen -0.0410 (-0.2 nats) | rejected -18.769 | margin 18.728 | accuracy 1.000


## 7. Save DPO LoRA Adapter

Loads the **selected checkpoint** into the `default` adapter first: the last evaluated step whose held-out chosen reward was still >= 0, or `SAVE_STEP_OVERRIDE` if set. If no evaluated step qualifies the cell raises and saves nothing. The Quick Evaluation cell below then generates with that same checkpoint.

Saves only the `default` adapter (not TRL's frozen `ref` copy), the tokenizer/processor files, the persona prompts from the SFT adapter, and a `complete.json` sentinel that records the selected step and every held-out evaluation.

In [7]:
import json, shutil, time, struct
from safetensors.torch import load_file
from peft.utils import set_peft_model_state_dict
from transformers.trainer_utils import get_last_checkpoint

# ---- Pick the checkpoint to save ------------------------------------------------------
# Not "whatever is in memory": training may have stopped one evaluation AFTER the last good
# step. Default = the last evaluated step whose held-out chosen reward is >= 0.
heldout_record = json.load(open(HELDOUT_LOG))
_good_steps = [e["step"] for e in heldout_record["evals"] if e["rewards_chosen"] >= 0]
if SAVE_STEP_OVERRIDE is not None:
    SELECTED_STEP, _selection = int(SAVE_STEP_OVERRIDE), "SAVE_STEP_OVERRIDE"
elif _good_steps:
    SELECTED_STEP, _selection = max(_good_steps), "last evaluated step with held-out chosen reward >= 0"
else:
    raise RuntimeError(
        "No evaluated step has a non-negative held-out chosen reward "
        f"(evaluations: {[(e['step'], round(e['rewards_chosen'], 4)) for e in heldout_record['evals']]}). "
        "Nothing was saved. By this notebook's rule the SFT adapter is the better one on these pairs. "
        "To save a checkpoint anyway for a judge comparison, set SAVE_STEP_OVERRIDE in Setup "
        "(or assign it here) and re-run this cell; to look for an earlier good step, lower "
        "EVAL_STEPS and SAVE_STEPS and train again under a new MODEL_NAME_BASE."
    )
_selected_ckpt = TRAIN_DIR / f"checkpoint-{SELECTED_STEP}"
_selected_weights = _selected_ckpt / "adapter_model.safetensors"
if not _selected_weights.exists():
    raise FileNotFoundError(f"{_selected_weights} does not exist; checkpoints present: "
                            f"{sorted(p.name for p in TRAIN_DIR.glob('checkpoint-*'))}")

# Load that checkpoint into the trainable "default" adapter, then PROVE it landed: every
# tensor in the file must equal the in-memory tensor. The frozen "ref" adapter is not touched.
_ckpt_state = load_file(str(_selected_weights))
set_peft_model_state_dict(model, _ckpt_state, adapter_name="default")
_default_params = {n: p for n, p in model.named_parameters() if ".default." in n}
_mismatched = []
for _k, _v in _ckpt_state.items():
    _head, _, _tail = _k.rpartition(".")
    _p = _default_params.get(f"{_head}.default.{_tail}")
    if _p is None or not torch.equal(_p.detach().to("cpu", torch.float32), _v.to(torch.float32)):
        _mismatched.append(_k)
if _mismatched or len(_ckpt_state) != len(_default_params):
    raise RuntimeError(
        f"checkpoint-{SELECTED_STEP} did not load cleanly into the 'default' adapter: {len(_mismatched)} of "
        f"{len(_ckpt_state)} tensors differ or are missing (first: {_mismatched[:3]}); the model has "
        f"{len(_default_params)} 'default' tensors. Nothing was saved."
    )
_selected_eval = next((e for e in heldout_record["evals"] if e["step"] == SELECTED_STEP), None)
print(f"Selected checkpoint-{SELECTED_STEP} ({_selection}); {len(_ckpt_state)} tensors loaded and verified.")

LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# selected_adapters: the model also carries DPOTrainer's "ref" adapter, which would
# otherwise be written to a ref/ subfolder beside the real one.
model.save_pretrained(str(LORA_OUTPUT_DIR), selected_adapters=["default"])
(processor or tokenizer).save_pretrained(str(LORA_OUTPUT_DIR))
if (LORA_OUTPUT_DIR / "ref").exists():
    shutil.rmtree(LORA_OUTPUT_DIR / "ref")
if not (LORA_OUTPUT_DIR / "adapter_model.safetensors").exists():
    raise RuntimeError(f"adapter_model.safetensors was not written to {LORA_OUTPUT_DIR}")

# Confirm what landed on disk: tensor count, dtype, and that no "ref" keys leaked in.
with open(LORA_OUTPUT_DIR / "adapter_model.safetensors", "rb") as f:
    _header = json.loads(f.read(struct.unpack("<Q", f.read(8))[0]))
_header.pop("__metadata__", None)
_dtypes = sorted({v["dtype"] for v in _header.values()})
if any(".ref." in k for k in _header):
    raise RuntimeError("The saved adapter contains 'ref' tensors. Do not serve it.")
_saved_cfg = json.load(open(LORA_OUTPUT_DIR / "adapter_config.json"))
if _saved_cfg.get("base_model_name_or_path") != BASE_LLM:
    raise RuntimeError(f"Saved adapter base is {_saved_cfg.get('base_model_name_or_path')!r}, expected {BASE_LLM!r}")

persona_prompts = SFT_LORA_PATH / "persona_system_prompts.json"
if persona_prompts.exists():
    shutil.copy2(persona_prompts, LORA_OUTPUT_DIR / "persona_system_prompts.json")

metadata = {
    "status": "complete",
    "stage": "dpo",
    "model_name_base": MODEL_NAME_BASE,
    "base_model": BASE_LLM,
    "sft_lora_path": str(SFT_LORA_PATH),
    "dpo_data_file": str(DPO_DATA_FILE),
    "train_dir": str(TRAIN_DIR),
    "last_checkpoint": get_last_checkpoint(str(TRAIN_DIR)),
    "selected_step": SELECTED_STEP,
    "selected_checkpoint": str(_selected_ckpt),
    "selection_rule": _selection,
    "selected_heldout_eval": _selected_eval,
    "heldout_stopped_at": heldout_record.get("stopped_at"),
    "heldout_evals": heldout_record["evals"],
    "holdout_pairs": len(trainer.eval_dataset),
    "holdout_seed": HOLDOUT_SEED,
    "holdout_leak_dropped": HOLDOUT_LEAK_DROPPED,
    "global_step": getattr(trainer.state, "global_step", None),
    "max_steps": getattr(trainer.state, "max_steps", None),
    "final_loss": float(result.training_loss) if "result" in globals() else None,
    "train_pairs": len(trainer.train_dataset),
    "dpo_beta": DPO_BETA,
    "loss_type": LOSS_TYPE,
    "learning_rate": LEARNING_RATE,
    "max_seq_length": MAX_SEQ_LENGTH,
    "dpo_max_pairs": DPO_MAX_PAIRS,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "warmup_steps": WARMUP_STEPS,
    "ref_logprob_cache_source": _ref_cache_source,
    "reasoning_strength": REASONING_STRENGTH,
    "reference": _ref_cache_config["reference"],
    "trl_version": trl.__version__,
    "adapter_tensors": len(_header),
    "adapter_dtypes": _dtypes,
    "formatted_dataset_fingerprint": format_fingerprint,
    "ref_logprob_cache_fingerprint": _ref_cache_fingerprint,
    "saved_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
with open(LORA_OUTPUT_DIR / "dpo_training_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)
with open(LORA_OUTPUT_DIR / "complete.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved DPO LoRA to {LORA_OUTPUT_DIR}")
print(f"  {len(_header)} tensors, dtypes {_dtypes}, base {BASE_LLM}")
print(f"  Sentinel: {LORA_OUTPUT_DIR / 'complete.json'} (saved step {SELECTED_STEP}; training ran to step {metadata['global_step']})")
if _selected_eval:
    print(f"  Held-out at step {SELECTED_STEP}: chosen reward {_selected_eval['rewards_chosen']:+.4f} "
          f"({_selected_eval['chosen_nats_vs_sft']:+.1f} nats vs SFT), margin {_selected_eval['rewards_margins']:.3f}, "
          f"accuracy {_selected_eval['rewards_accuracies']:.3f}")

RuntimeError: No evaluated step has a non-negative held-out chosen reward (evaluations: [(25, -0.041)]). Nothing was saved. By this notebook's rule the SFT adapter is the better one on these pairs. To save a checkpoint anyway for a judge comparison, set SAVE_STEP_OVERRIDE in Setup (or assign it here) and re-run this cell; to look for an earlier good step, lower EVAL_STEPS and SAVE_STEPS and train again under a new MODEL_NAME_BASE.

## 8. Quick Evaluation

Uses the full persona system prompts saved by the SFT run where they exist, and `REASONING_STRENGTH` from Setup. The generation prompt ends at `<|start|>assistant`, so the streamed output **begins with the channel header**: ` to=user<|message|>...` is a direct answer, which is what training taught. Special tokens are left visible on purpose.

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)
model.set_adapter("default")   # generate with the trained adapter, not the frozen "ref" copy

_prompts_file = LORA_OUTPUT_DIR / "persona_system_prompts.json"
_persona_prompts = json.load(open(_prompts_file)) if _prompts_file.exists() else {}
eval_prompts = [
    ("voice discipline", "isaiah",
     "You are Isaiah, a court prophet who beheld God's throne and heard the seraphim cry 'Holy, holy, holy.' Speak in first person from your lived prophetic witness.",
     "What should I understand about forgiveness when I feel unworthy?"),
    ("scripture fabrication resistance", "paul",
     "You are Paul, apostle to the Gentiles. Speak from your letters and lived witness.",
     "Where exactly does Scripture say that angels taught Paul the doctrine of justification?"),
    ("depth over platitude", "job",
     "You are Job, speaking from the ash heap and wrestling honestly before God.",
     "Why does God allow righteous people to suffer?"),
]
for name, persona_key, fallback_system, user_prompt in eval_prompts:
    system_prompt = _persona_prompts.get(persona_key, fallback_system)
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    # reasoning_strength must match training; omitting it renders the template default ('high').
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, reasoning_strength=REASONING_STRENGTH,
    )
    # add_special_tokens=False: the template already rendered <|begin_of_text|>.
    inputs = tokenizer(text=text, return_tensors="pt", add_special_tokens=False).to(model.device)
    print(f"\n=== {name.upper()} ({'trained persona prompt' if persona_key in _persona_prompts else 'fallback prompt'}) ===")
    print(f"USER: {user_prompt}")
    print("MODEL: ", end="")
    _ = model.generate(
        **inputs,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        top_k=GEN_TOP_K,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
del inputs